# 02 — Randomized Experiment Analysis & ATE

## Objective
Quantify the **average causal effect** of Men's Email, Women's Email, and Any Email relative to No Email.

Outcomes:
- visit
- conversion
- spend

The notebook reports:
- treatment/control rates
- absolute lift
- relative lift
- bootstrap confidence intervals
- two-sided permutation/randomization p-values
- treatment-group comparisons

In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

# Find the project root so the notebook works whether launched from the project root
# or from inside the notebooks directory.
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
ROOT = next((p for p in _candidates if (p / "src" / "config.py").exists()), _here)
if not (ROOT / "src" / "config.py").exists():
    raise FileNotFoundError("Could not locate project root containing src/config.py")

sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
from src.data_utils import load_data, prepare
from src.config import DATA_PATH

df = prepare(load_data(DATA_PATH))
print(df.shape)
display(df.groupby("segment")[["visit","conversion","spend"]].mean())

## 1. ATE helper functions

In [ ]:
def bootstrap_ci(a, b, stat=np.mean, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    a = np.asarray(a)
    b = np.asarray(b)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        sa = rng.choice(a, len(a), replace=True)
        sb = rng.choice(b, len(b), replace=True)
        diffs[i] = stat(sa) - stat(sb)
    return np.quantile(diffs, [0.025, 0.975])

def permutation_pvalue(a, b, n_perm=3000, seed=42):
    rng = np.random.default_rng(seed)
    a = np.asarray(a)
    b = np.asarray(b)
    observed = a.mean() - b.mean()
    pooled = np.concatenate([a, b])
    n_a = len(a)
    extreme = 0
    for _ in range(n_perm):
        shuffled = rng.permutation(pooled)
        diff = shuffled[:n_a].mean() - shuffled[n_a:].mean()
        if abs(diff) >= abs(observed):
            extreme += 1
    return (extreme + 1) / (n_perm + 1)

def ate_row(df, treatment, control="No E-Mail", outcome="conversion"):
    a = df.loc[df.segment == treatment, outcome].astype(float).to_numpy()
    b = df.loc[df.segment == control, outcome].astype(float).to_numpy()
    lift = a.mean() - b.mean()
    relative = lift / b.mean() if b.mean() != 0 else np.nan
    lo, hi = bootstrap_ci(a, b)
    p = permutation_pvalue(a, b)
    return {
        "treatment": treatment, "control": control, "outcome": outcome,
        "treatment_mean": a.mean(), "control_mean": b.mean(),
        "absolute_lift": lift, "relative_lift": relative,
        "ci_low": lo, "ci_high": hi, "p_value": p,
        "treatment_n": len(a), "control_n": len(b)
    }

## 2. Men's Email vs Control and Women's Email vs Control

In [ ]:
rows = []
for arm in ["Mens E-Mail", "Womens E-Mail"]:
    for outcome in ["visit", "conversion", "spend"]:
        rows.append(ate_row(df, arm, outcome=outcome))
ate = pd.DataFrame(rows)
display(
    ate.style.format({
        "treatment_mean":"{:.6f}", "control_mean":"{:.6f}",
        "absolute_lift":"{:.6f}", "relative_lift":"{:.2%}",
        "ci_low":"{:.6f}", "ci_high":"{:.6f}", "p_value":"{:.4f}"
    })
)
ate.to_csv(ROOT / "artifacts" / "metrics" / "notebook_02_ate_results.csv", index=False)

## 3. Any Email vs Control

In [ ]:
df_any = df.copy()
df_any["any_segment"] = np.where(df_any["segment"] == "No E-Mail", "No E-Mail", "Any Email")
rows_any = []
for outcome in ["visit", "conversion", "spend"]:
    a = df_any.loc[df_any.any_segment == "Any Email", outcome].to_numpy()
    b = df_any.loc[df_any.any_segment == "No E-Mail", outcome].to_numpy()
    rows_any.append({
        "treatment":"Any Email", "control":"No E-Mail", "outcome":outcome,
        "treatment_mean":a.mean(), "control_mean":b.mean(),
        "absolute_lift":a.mean()-b.mean(),
        "relative_lift":(a.mean()-b.mean())/b.mean() if b.mean() else np.nan,
        "ci_low":bootstrap_ci(a,b)[0], "ci_high":bootstrap_ci(a,b)[1],
        "p_value":permutation_pvalue(a,b)
    })
ate_any = pd.DataFrame(rows_any)
display(ate_any.style.format({
    "treatment_mean":"{:.6f}", "control_mean":"{:.6f}",
    "absolute_lift":"{:.6f}", "relative_lift":"{:.2%}",
    "ci_low":"{:.6f}", "ci_high":"{:.6f}", "p_value":"{:.4f}"
}))
ate_any.to_csv(ROOT / "artifacts" / "metrics" / "notebook_02_any_email_ate.csv", index=False)

## 4. Treatment ranking by conversion lift

In [ ]:
conv_ate = pd.concat([
    ate[ate.outcome=="conversion"][["treatment","absolute_lift","relative_lift","p_value"]],
    ate_any[ate_any.outcome=="conversion"][["treatment","absolute_lift","relative_lift","p_value"]]
]).sort_values("absolute_lift", ascending=False)
display(conv_ate)

## Interpretation
ATE answers the population-level question: **does assigning this campaign arm change average outcomes?**

It does not tell us which individual customers are likely to benefit. That is the purpose of the next stages: heterogeneous treatment-effect estimation and uplift modeling.